# Q9: do text states seed, maintain, or read back from image registers?
Choose an A100 GPU runtime. Start with **smoke**, then **discovery/screen**, and freeze promising contrasts before **confirm**. Screening replays individual transformer forwards from the clean trajectory; only confirm finishes edited images.
FLUX.1-dev and Schnell are supported. PixArt has no evolving DiT text stream for the read-back test. Raw activations and replay states stay in memory; full images stay in `/content`. Optional Drive export is limited to 250 MiB of compact artifacts per run.
See [SPEC_Q9.md](https://github.com/BrendanGho/massive-activations-fig3/blob/main/SPEC_Q9.md) for the design and interpretation limits.

In [ ]:
# Install once in a fresh runtime.
import os, subprocess, sys
REPO_URL = 'https://github.com/BrendanGho/massive-activations-fig3.git' # @param {type:"string"}
BRANCH = 'main' # @param {type:"string"}
REPO_DIR = '/content/massive-activations-fig3'
if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only', 'origin', BRANCH], check=True)
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path: sys.path.insert(0, REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[q9]'], check=True)
import torch
assert torch.cuda.is_available(), 'Select a GPU runtime before running Q9.'
print(torch.cuda.get_device_name(), 'VRAM GiB:', torch.cuda.get_device_properties(0).total_memory / 2**30)
from huggingface_hub import login
from google.colab import userdata
try:
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    hf_token = None
if hf_token:
    login(token=hf_token, add_to_git_credential=False)
else:
    login(add_to_git_credential=False)
# Accept the FLUX.1-dev license on Hugging Face before loading the gated checkpoint.


In [ ]:
# Model-specific values come from Q9 PRESETS in the repository.
import json
from dataclasses import asdict
from pathlib import Path
from src.experiments.text_image_coupling import preset_config, PRESETS, EDGE_METHODS
Q9_MODEL = 'flux1-dev' # @param ['flux1-dev', 'flux-schnell']
Q9_MODE = 'smoke' # @param ['smoke', 'discovery', 'screen', 'confirm']
Q9_RESOLUTION = 1024 # @param {type:"integer"}
Q9_SITES = '' # @param {type:"string"}
Q9_STEPS = '' # @param {type:"string"}
Q9_METHODS = '' # @param {type:"string"}
Q9_CANDIDATE_CLASSES = 'eos,pad' # @param {type:"string"}
Q9_CANDIDATE_SOURCE = 'union' # @param ['norm', 'sink', 'union', 'intersection']
Q9_FORCE_OFFLOAD = False # @param {type:"boolean"}
Q9_STRUCTURED_SCORES = '' # @param {type:"string"}
cfg = preset_config(Q9_MODEL, Q9_MODE)
cfg.resolution = Q9_RESOLUTION
cfg.dtype = 'bf16' if torch.cuda.is_bf16_supported() else 'fp16'
cfg.offload = Q9_FORCE_OFFLOAD or torch.cuda.get_device_properties(0).total_memory / 2**30 < 38
cfg.candidate_classes = [x.strip() for x in Q9_CANDIDATE_CLASSES.split(',') if x.strip()]
cfg.candidate_source = Q9_CANDIDATE_SOURCE
if Q9_SITES.strip(): cfg.sites = [int(x) for x in Q9_SITES.split(',')]
if Q9_STEPS.strip(): cfg.steps = [int(x) for x in Q9_STEPS.split(',')]
if Q9_METHODS.strip(): cfg.methods = [x.strip() for x in Q9_METHODS.split(',')]
cfg.structured_scores = Q9_STRUCTURED_SCORES or None
# Advanced controls: edit cfg.prompts, cfg.seeds, cfg.rescue_layer, cfg.rescues,
# cfg.attention_layers, and calibration prompts here before saving.
# sites=-1 targets projected T5 inputs; sites>=0 target post-block text states.
cfg.validate()
jobs_per_pair = sum(1 for l in cfg.sites for t in cfg.steps for m in cfg.methods for r in cfg.rescues
                    if (r == 'none' or m == 'remove_direction')
                    and not (l == -1 and (m in EDGE_METHODS or m.startswith('image_'))))
print('Model:', PRESETS[Q9_MODEL], 'offload:', cfg.offload)
print('Clean calibration trajectories if uncached:', len(cfg.calibration_prompts)*len(cfg.calibration_seeds))
print('Evaluation pairs:', len(cfg.prompts)*len(cfg.seeds), 'jobs per pair:', jobs_per_pair)
print('Jobs are full trajectories in confirm; single transformer forwards in smoke/screen.')
print('Native empty-prompt baselines:', len(cfg.seeds) if cfg.include_empty else 0)
Q9_CONFIG_PATH = '/content/q9_config.json'
Path(Q9_CONFIG_PATH).write_text(json.dumps(asdict(cfg), indent=2))
print(json.dumps(asdict(cfg), indent=2))


In [ ]:
# A subprocess releases model memory when finished; output appears live.
subprocess.run([sys.executable, '-u', '-m', 'src.experiments.text_image_coupling',
                '--config', Q9_CONFIG_PATH], check=True)


In [ ]:
from IPython.display import display, Image
from src.experiments.q9_report import locate
q9_result = locate(cfg)
print((q9_result / 'report_status.json').read_text())
for figure in sorted((q9_result / 'figures').glob('*.png')):
    print(figure.name)
    display(Image(filename=str(figure)))
print('No-candidate or unavailable-control conditions are excluded from causal summaries.')
print('Check audits.csv and direction_stability.csv before interpreting a negative result.')


In [ ]:
Q9_EXPORT_TO_DRIVE = False # @param {type:"boolean"}
Q9_DRIVE_ROOT = '/content/drive/MyDrive/Research/MA/q9_compact' # @param {type:"string"}
if Q9_EXPORT_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    subprocess.run([sys.executable, '-m', 'src.experiments.text_image_coupling',
                    '--config', Q9_CONFIG_PATH, '--export-compact', Q9_DRIVE_ROOT], check=True)
else:
    print('All results remain in temporary Colab storage:', q9_result)
